# Task 6 — RINE-style Stage B

Clean-only ablation on fixed-Q96 using the same splits and seeds as Stage A. CLIP stays frozen; only layer importance and the binary head train. Final retention remains pending Task 3 robustness cells.

In [ ]:
# 1. GPU and Drive.
import torch
assert torch.cuda.is_available(), 'Task 6 requires a GPU Colab server'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Refresh repository and dependencies.
from pathlib import Path
import subprocess
import sys
import shutil

PROJECT_ROOT = Path('/content/cya-techjam26')
REPOSITORY_URL = 'https://github.com/maxi-cmyk/cya-techjam26.git'
if (PROJECT_ROOT / '.git').is_dir():
    status = subprocess.run(['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, check=True, capture_output=True, text=True).stdout.splitlines()
    unexpected = [line for line in status if not line.endswith('configs/colab.json')]
    assert not unexpected, f'Unexpected remote checkout changes: {unexpected}'
    if status:
        subprocess.run(['git', 'restore', 'configs/colab.json'], cwd=PROJECT_ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_ROOT, check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], cwd=PROJECT_ROOT, check=True)

In [ ]:
# 3. Restore Task 2 inputs and Stage A comparison records from Drive.
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
DRIVE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
TASK2_ROOT = ARTIFACT_ROOT / 'task2'
input_archive = DRIVE_ARTIFACT_ROOT / 'task2_stagea_bundle.tar.gz'
assert input_archive.is_file(), input_archive
TASK2_ROOT.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(input_archive, TASK2_ROOT)
manifest = TASK2_ROOT / 'fixed_q96_manifest.csv'
assert manifest.is_file(), manifest
drive_stage_a = DRIVE_ARTIFACT_ROOT / 'task4/fixed_q96'
local_stage_a = ARTIFACT_ROOT / 'task4/fixed_q96'
assert drive_stage_a.is_dir(), drive_stage_a
shutil.copytree(drive_stage_a, local_stage_a, dirs_exist_ok=True)
for seed in (42, 43, 44):
    assert (local_stage_a / f'seed_{seed}/best_clean_predictions.csv').is_file()
print('Task 6 inputs ready')

In [ ]:
# 4. Pin the exact model commit used by Stage A.
import json
from huggingface_hub import model_info
config_path = PROJECT_ROOT / 'configs/colab.json'
config = json.loads(config_path.read_text())
resolved_commit = model_info(config['model']['identifier'], revision=config['model']['revision']).sha
config['model']['revision'] = resolved_commit
config_path.write_text(json.dumps(config, indent=2) + '\n')
print('Layers:', config['model']['rine_layers'])
print('Commit:', resolved_commit)

In [ ]:
# 5. Resumable Stage B runner.
import os
RINE_CACHE = Path('/content/rine_feature_cache')
seeds = (42, 43, 44)

def run_rine(seed):
    local_run = ARTIFACT_ROOT / 'task6/fixed_q96' / f'seed_{seed}'
    drive_run = DRIVE_ARTIFACT_ROOT / 'task6/fixed_q96' / f'seed_{seed}'
    if not (local_run / 'training_summary.json').is_file() and drive_run.is_dir():
        shutil.copytree(drive_run, local_run, dirs_exist_ok=True)
    if (local_run / 'training_summary.json').is_file():
        print(f'SKIP complete: seed {seed}')
        return
    print(f'RUN RINE: seed {seed}', flush=True)
    subprocess.run([
        sys.executable, 'scripts/train_rine_baseline.py',
        '--manifest', str(manifest),
        '--matching-policy', 'fixed_q96',
        '--output-root', str(ARTIFACT_ROOT / 'task6'),
        '--cache-root', str(RINE_CACHE),
        '--seed', str(seed),
        '--physical-batch-size', '4',
    ], cwd=PROJECT_ROOT, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    shutil.copytree(local_run, drive_run, dirs_exist_ok=True)
    print('SAVED:', drive_run)

In [ ]:
# 6. First proof run; inspect before continuing.
run_rine(42)
first = ARTIFACT_ROOT / 'task6/fixed_q96/seed_42'
extraction = json.loads((first / 'extraction_report.json').read_text())
training = json.loads((first / 'training_summary.json').read_text())
print(json.dumps({
    'extraction': extraction,
    'best_clean_accuracy': training['best_values']['clean'],
    'best_layer_importance': training['best_clean_layer_importance'],
}, indent=2))

In [ ]:
# 7. Remaining seeds reuse the cached intermediate features.
for seed in seeds:
    run_rine(seed)

In [ ]:
# 8. Apples-to-apples clean comparison; final decision remains pending Task 3.
comparison_path = ARTIFACT_ROOT / 'task6/stage_a_vs_rine.json'
subprocess.run([
    sys.executable, 'scripts/compare_stage_a_rine.py',
    '--stage-a-root', str(ARTIFACT_ROOT / 'task4'),
    '--stage-b-root', str(ARTIFACT_ROOT / 'task6'),
    '--output', str(comparison_path),
], cwd=PROJECT_ROOT, check=True)
drive_comparison = DRIVE_ARTIFACT_ROOT / 'task6/stage_a_vs_rine.json'
drive_comparison.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(comparison_path, drive_comparison)
comparison = json.loads(comparison_path.read_text())
print('Provisional decision:', comparison['provisional_clean_only_decision'])
print('Final decision:', comparison['final_decision'])
print(json.dumps(comparison['aggregate'], indent=2))